In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- DIM: gold.dim_lead
-- =====================================================

CREATE TABLE IF NOT EXISTS gold.dim_lead
(
    lead_id INT,

    lead_name STRING,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

-- MERGE

MERGE INTO gold.dim_lead AS target

USING
(
    SELECT
        crm_lead_id AS lead_id,
        name        AS lead_name,
        created_at,
        updated_at,
        current_timestamp() AS dwh_loaded_at
    FROM silver.crm_lead
    WHERE updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.dim_lead),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.lead_id = source.lead_id

WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.lead_name     = source.lead_name,
    target.created_at    = source.created_at,
    target.updated_at    = source.updated_at,
    target.dwh_loaded_at = current_timestamp()

WHEN NOT MATCHED THEN INSERT *;

-- =====================================================
-- DIM: gold.dim_stage
-- =====================================================

CREATE TABLE IF NOT EXISTS gold.dim_stage
(
    stage_id INT,

    stage_name STRING,
    is_won BOOLEAN,
    fold BOOLEAN,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

-- MERGE

MERGE INTO gold.dim_stage AS target

USING
(
    SELECT
        crm_stage_id AS stage_id,
        name         AS stage_name,
        is_won,
        fold,
        current_timestamp() AS dwh_loaded_at
    FROM silver.crm_stage
    WHERE updated_at > COALESCE
    (
        (SELECT MAX(dwh_loaded_at) FROM gold.dim_stage),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.stage_id = source.stage_id

WHEN MATCHED THEN
UPDATE SET
    target.stage_name    = source.stage_name,
    target.is_won        = source.is_won,
    target.fold          = source.fold,
    target.dwh_loaded_at = current_timestamp()

WHEN NOT MATCHED THEN INSERT *;

-- =====================================================
-- DIM: gold.dim_team
-- =====================================================

CREATE TABLE IF NOT EXISTS gold.dim_team
(
    team_id INT,

    team_name STRING,
    company_id INT,
    company_name STRING,

    invoiced_target DOUBLE,

    active BOOLEAN,
    use_leads BOOLEAN,
    use_opportunities BOOLEAN,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

-- MERGE

MERGE INTO gold.dim_team AS target

USING
(
    SELECT
        crm_team_id AS team_id,
        name        AS team_name,
        company_id,
        company_name,
        invoiced_target,
        active,
        use_leads,
        use_opportunities,
        current_timestamp() AS dwh_loaded_at
    FROM silver.crm_team
    WHERE updated_at > COALESCE
    (
        (SELECT MAX(dwh_loaded_at) FROM gold.dim_team),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.team_id = source.team_id

WHEN MATCHED THEN
UPDATE SET
    target.team_name         = source.team_name,
    target.company_id        = source.company_id,
    target.company_name      = source.company_name,
    target.invoiced_target   = source.invoiced_target,
    target.active            = source.active,
    target.use_leads         = source.use_leads,
    target.use_opportunities = source.use_opportunities,
    target.dwh_loaded_at     = current_timestamp()

WHEN NOT MATCHED THEN INSERT *;

-- =====================================================
-- DIM: gold.dim_campaign
-- =====================================================

CREATE TABLE IF NOT EXISTS gold.dim_campaign
(
    campaign_id INT,

    campaign_name STRING,
    company_id INT,
    company_name STRING,

    active BOOLEAN,
    is_auto_campaign BOOLEAN,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

-- MERGE

MERGE INTO gold.dim_campaign AS target

USING
(
    SELECT
        utm_campaign_id AS campaign_id,
        name            AS campaign_name,
        company_id,
        company_name,
        active,
        is_auto_campaign,
        current_timestamp() AS dwh_loaded_at
    FROM silver.utm_campaign
    WHERE updated_at > COALESCE
    (
        (SELECT MAX(dwh_loaded_at) FROM gold.dim_campaign),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.campaign_id = source.campaign_id

WHEN MATCHED THEN
UPDATE SET
    target.campaign_name     = source.campaign_name,
    target.company_id        = source.company_id,
    target.company_name      = source.company_name,
    target.active            = source.active,
    target.is_auto_campaign  = source.is_auto_campaign,
    target.dwh_loaded_at     = current_timestamp()

WHEN NOT MATCHED THEN INSERT *;

-- =====================================================
-- DIM: gold.dim_lost_reason
-- =====================================================

CREATE TABLE IF NOT EXISTS gold.dim_lost_reason
(
    lost_reason_id INT,

    lost_reason_name STRING,
    active BOOLEAN,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

-- MERGE

MERGE INTO gold.dim_lost_reason AS target

USING
(
    SELECT
        crm_lost_reason_id AS lost_reason_id,
        name               AS lost_reason_name,
        active,
        current_timestamp() AS dwh_loaded_at
    FROM silver.crm_lost_reason
    WHERE updated_at > COALESCE
    (
        (SELECT MAX(dwh_loaded_at) FROM gold.dim_lost_reason),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.lost_reason_id = source.lost_reason_id

WHEN MATCHED THEN
UPDATE SET
    target.lost_reason_name = source.lost_reason_name,
    target.active           = source.active,
    target.dwh_loaded_at    = current_timestamp()

WHEN NOT MATCHED THEN INSERT *;

-- =====================================================
-- FACT: gold.fact_crm_pipeline
-- =====================================================

CREATE TABLE IF NOT EXISTS gold.fact_crm_pipeline
(
    lead_id INT,

    stage_id INT,
    stage_name STRING,

    team_id INT,
    team_name STRING,

    campaign_id INT,
    campaign_name STRING,

    lost_reason_id INT,
    lost_reason_name STRING,

    partner_id INT,
    partner_name STRING,

    company_id INT,
    company_name STRING,

    lead_name STRING,
    type STRING,
    priority STRING,
    won_status STRING,

    expected_revenue DOUBLE,
    prorated_revenue DOUBLE,
    recurring_revenue_monthly DOUBLE,
    probability DOUBLE,
    automated_probability DOUBLE,

    day_open DOUBLE,
    day_close DOUBLE,

    is_won BOOLEAN,
    is_lost BOOLEAN,
    active BOOLEAN,

    date_deadline DATE,
    date_open TIMESTAMP,
    date_closed TIMESTAMP,
    date_conversion TIMESTAMP,
    date_last_stage_update TIMESTAMP,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

-- MERGE

MERGE INTO gold.fact_crm_pipeline AS target

USING
(
    SELECT
        l.crm_lead_id       AS lead_id,

        l.stage_id,
        s.name              AS stage_name,

        l.team_id,
        t.name              AS team_name,

        l.campaign_id,
        c.name              AS campaign_name,

        l.lost_reason_id,
        lr.name             AS lost_reason_name,

        l.partner_id,
        l.partner_name,

        l.company_id,
        l.company_name,

        l.name              AS lead_name,
        l.type,
        l.priority,
        l.won_status,

        l.expected_revenue,
        l.prorated_revenue,
        l.recurring_revenue_monthly,
        l.probability,
        l.automated_probability,

        l.day_open,
        l.day_close,

        s.is_won                                AS is_won,
        CASE WHEN l.won_status = 'lost'
             THEN TRUE ELSE FALSE END           AS is_lost,
        l.active,

        CAST(l.date_deadline AS DATE)           AS date_deadline,
        l.date_open,
        l.date_closed,
        l.date_conversion,
        l.date_last_stage_update,

        l.created_at,
        l.updated_at,

        current_timestamp() AS dwh_loaded_at

    FROM silver.crm_lead l
    LEFT JOIN silver.crm_stage s
        ON s.crm_stage_id = l.stage_id
    LEFT JOIN silver.crm_team t
        ON t.crm_team_id = l.team_id
    LEFT JOIN silver.utm_campaign c
        ON c.utm_campaign_id = l.campaign_id
    LEFT JOIN silver.crm_lost_reason lr
        ON lr.crm_lost_reason_id = l.lost_reason_id

    WHERE l.updated_at > COALESCE
    (
        (SELECT MAX(updated_at) FROM gold.fact_crm_pipeline),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source

ON target.lead_id = source.lead_id

WHEN MATCHED AND target.updated_at < source.updated_at THEN
UPDATE SET
    target.stage_id                  = source.stage_id,
    target.stage_name                = source.stage_name,
    target.team_id                   = source.team_id,
    target.team_name                 = source.team_name,
    target.campaign_id               = source.campaign_id,
    target.campaign_name             = source.campaign_name,
    target.lost_reason_id            = source.lost_reason_id,
    target.lost_reason_name          = source.lost_reason_name,
    target.partner_id                = source.partner_id,
    target.partner_name              = source.partner_name,
    target.company_id                = source.company_id,
    target.company_name              = source.company_name,
    target.lead_name                 = source.lead_name,
    target.type                      = source.type,
    target.priority                  = source.priority,
    target.won_status                = source.won_status,
    target.expected_revenue          = source.expected_revenue,
    target.prorated_revenue          = source.prorated_revenue,
    target.recurring_revenue_monthly = source.recurring_revenue_monthly,
    target.probability               = source.probability,
    target.automated_probability     = source.automated_probability,
    target.day_open                  = source.day_open,
    target.day_close                 = source.day_close,
    target.is_won                    = source.is_won,
    target.is_lost                   = source.is_lost,
    target.active                    = source.active,
    target.date_deadline             = source.date_deadline,
    target.date_open                 = source.date_open,
    target.date_closed               = source.date_closed,
    target.date_conversion           = source.date_conversion,
    target.date_last_stage_update    = source.date_last_stage_update,
    target.updated_at                = source.updated_at,
    target.dwh_loaded_at             = current_timestamp()

WHEN NOT MATCHED THEN INSERT *;